# 辅助视觉设备渲染管线（Notebook 版）

这个 Notebook 对应仓库中的渲染管线实现，适合在 Jupyter Notebook 中逐步运行与调参。

## 1. 环境准备
- Notebook 会自动向上查找项目根目录（包含 `src/biopiccw/pipeline.py`）。
- 无论你从仓库根目录还是 `notebooks/` 目录启动 Jupyter，均可导入 `biopiccw.pipeline`。

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'biopiccw' / 'pipeline.py').exists():
            return candidate
    # 回退到当前目录（若用户在其它位置打开 notebook）
    return current

PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
for p in (PROJECT_ROOT, SRC):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print('Working directory:', Path.cwd())
print('Detected project root:', PROJECT_ROOT)
print('src in sys.path:', str(SRC) in sys.path)


## 2. 在 Notebook 内实现核心渲染函数（独立运行）
下面直接在 Notebook 中定义完整渲染管线函数，便于调试和理解，不依赖 `biopiccw.pipeline` 外部导入。

In [ ]:
import time
from pathlib import Path
from PIL import Image, ImageEnhance

def load_image(image_path):
    """加载图像并统一转换为 RGB。"""
    return Image.open(image_path).convert("RGB")

def apply_magnification(image, magnification_factor):
    """按倍率放大图像。"""
    if magnification_factor <= 0:
        raise ValueError("magnification_factor must be > 0")
    width, height = image.size
    new_size = (int(width * magnification_factor), int(height * magnification_factor))
    new_size = (max(1, new_size[0]), max(1, new_size[1]))
    return image.resize(new_size, Image.Resampling.LANCZOS)

def enhance_contrast(image, contrast_factor):
    """增强图像对比度。"""
    if contrast_factor < 0:
        raise ValueError("contrast_factor must be >= 0")
    enhancer = ImageEnhance.Contrast(image)
    return enhancer.enhance(contrast_factor)

def sharpen_image(image, sharpness_factor):
    """增强图像锐度。"""
    if sharpness_factor < 0:
        raise ValueError("sharpness_factor must be >= 0")
    enhancer = ImageEnhance.Sharpness(image)
    return enhancer.enhance(sharpness_factor)

def adjust_dynamic_range(image, gamma):
    """伽马校正，支持多通道 LUT（如 RGB 需要 256*3）。"""
    if gamma <= 0:
        raise ValueError("gamma must be > 0")
    base_lut = [min(255, int((x / 255) ** (1 / gamma) * 255)) for x in range(256)]
    bands = len(image.getbands()) if hasattr(image, "getbands") else 1
    lut = base_lut * max(1, bands)
    return image.point(lut)

def simulate_latency(image, delay_seconds):
    """模拟设备处理延迟。"""
    if delay_seconds < 0:
        raise ValueError("delay_seconds must be >= 0")
    time.sleep(delay_seconds)
    return image

def render_pipeline(image_path, magnification_factor, contrast_factor, sharpness_factor, gamma, delay_seconds):
    """执行完整渲染管线。"""
    image = load_image(image_path)
    image = apply_magnification(image, magnification_factor)
    image = enhance_contrast(image, contrast_factor)
    image = sharpen_image(image, sharpness_factor)
    image = adjust_dynamic_range(image, gamma)
    image = simulate_latency(image, delay_seconds)
    return image

def save_image(image, output_path):
    """保存图像到磁盘。"""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path)

print("Notebook core functions are ready (standalone mode).")


## 3. 构造示例输入图像
为了保证 Notebook 可直接运行，这里先生成一张简单的测试图。

In [ ]:
demo_dir = PROJECT_ROOT / 'notebooks' / 'demo_outputs'
demo_dir.mkdir(parents=True, exist_ok=True)

input_path = demo_dir / 'input_scene.jpg'
output_path = demo_dir / 'output_scene.jpg'

Image.new('RGB', (80, 60), color=(120, 80, 40)).save(input_path)
print('Input image generated:', input_path)


## 4. 设置参数并执行渲染

In [ ]:
magnification_factor = 2.0
contrast_factor = 1.5
sharpness_factor = 2.0
gamma = 2.2
delay_seconds = 0.0

output_image = render_pipeline(
    image_path=input_path,
    magnification_factor=magnification_factor,
    contrast_factor=contrast_factor,
    sharpness_factor=sharpness_factor,
    gamma=gamma,
    delay_seconds=delay_seconds,
)

save_image(output_image, output_path)
print('Rendered image saved to', output_path)
print('Output size:', output_image.size)

## 5. 查看结果（在支持的 Jupyter 环境中显示）

In [ ]:
# 如果你的环境安装了 IPython.display，可取消注释直接显示图像
# from IPython.display import display
# display(output_image)

print('Input path :', input_path)
print('Output path:', output_path)

## 6. 参数校验示例（可选）
下面示例展示非法参数会抛出 `ValueError`。

In [ ]:
try:
    _ = render_pipeline(
        image_path=input_path,
        magnification_factor=0,
        contrast_factor=1.0,
        sharpness_factor=1.0,
        gamma=1.0,
        delay_seconds=0.0,
    )
except ValueError as e:
    print('Caught expected error:', e)